# 04 · A radar return is not a vessel label

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/blob/145a73e500cb06070f399a040206802cc279174e/content/04_sar_detection.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/04_sar_detection.ipynb` at `145a73e500cb06070f399a040206802cc279174e`; original SHA-256 `5ef4af8643a216e664e687e9a67ed54528d54d5f34bbad3b7796d53f2d1fb14f`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 04 · A radar return is not a vessel label

[Run this notebook in your browser ↗](https://jltobias.github.io/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/lab/index.html?path=04_sar_detection.ipynb)

**Evidence: a real calibrated Sentinel-1 crop, followed by a separately labeled synthetic detection experiment.** The real crop has no validated vessel annotations. The simulation supplies known target positions so we can evaluate detector behavior.

Sources: Copernicus Sentinel data (2024), processed through Microsoft Planetary Computer; [Sentinel-1 processing background](https://developers.google.com/earth-engine/guides/sentinel1). See `data/sentinel1_real_window.provenance.json` for the scene, acquisition, coordinate reference system and exact transformation.

Run the setup cell first. Browser Python downloads numerical packages on first use. Core exercises use bundled files; live access is optional. Export your notebook to preserve work.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "pandas", "matplotlib", "scipy", "plotly"])
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import oceanlab as lab
from IPython.display import display
print("Runtime:", sys.version.split()[0], "| NumPy:", np.__version__, "| pandas:", pd.__version__)

## Inspect actual radar pixels
The crop is a 512 × 512 window near Rostock harbor. The physical quantity is terrain-corrected **gamma0 linear intensity**. Convert positive values with `10 log10(I)` for display. This is not the sigma0 Earth Engine collection; different processing products should not be treated as interchangeable. The broad color stretch is illustrative. Bright returns can be buildings, infrastructure, vessels, or other structures.

In [ ]:
provenance=json.loads((lab.DATA/"sentinel1_real_window.provenance.json").read_text())
with np.load(lab.DATA/"sentinel1_real_window.npz") as archive:
    real=archive["gamma0_linear"]
real_db=10*np.log10(np.where(real>0,real,np.nan))
display(pd.Series({k:provenance[k] for k in ["item_id","acquired","band","units","center_lon_lat","shape"]}))
fig,ax=plt.subplots()
p=ax.imshow(real_db,cmap="gray",vmin=-25,vmax=5)
ax.set(xlabel="Source pixel column",ylabel="Source pixel row",title="OBSERVED SENTINEL-1 | Rostock coastal window, June 2024")
fig.colorbar(p,ax=ax,label="Gamma0 backscatter (dB)")
plt.show()
print(provenance["attribution"])

## Build an experiment with known targets
The remaining cells use **synthetic pixels**. Exponential speckle approximates a simple single-look intensity model; actual multilooked RTC statistics are more complex. Four bright targets are injected into a variable ocean background. A land mask excludes shoreline returns. Thresholding operates in linear intensity, not dB.

In [ ]:
ALPHA=7.0
image,land,truth=lab.radar()
candidates,background=lab.radar_detect(image,land,alpha=ALPHA)
display(pd.Series(lab.detection_metrics(candidates,truth)))
lab.radar_plot(ALPHA); plt.show()

## See the return in 3D
Height is backscatter in decibels, **not ship height**. Drag to inspect the injected target and surrounding speckle.

In [ ]:
lab.show(lab.radar_surface())

## Precision, recall and the price of a threshold
Candidates are local maxima. We match them one-to-one to injected target centers within three pixels. A duplicate candidate cannot count as a second true positive. These scores describe this toy image only; they do not estimate real-world ship detection performance.

In [ ]:
rows=[]
for alpha in [3,4,5,6,7,8,10,14,20,40,60,80]:
    points,_=lab.radar_detect(image,land,alpha=alpha)
    rows.append({"alpha":alpha,**lab.detection_metrics(points,truth)})
scores=pd.DataFrame(rows)
display(scores)
ax=scores.plot(x="alpha",y=["precision","recall"],marker="o",color=["#087e83","#bd6e13"])
ax.set(ylim=(-.05,1.05),ylabel="Object-level score",title="SYNTHETIC DETECTOR | threshold trade-off")
plt.show()

## What would make a real detector credible?
We would need a trustworthy land/infrastructure mask, sensor-specific calibration, labeled independent scenes, vessel-size and sea-state strata, coverage accounting, and validation of both false alarms and missed vessels. The toy detector is not applied to the real crop as a validated ship classifier. Inspect the actual crop and explain why a simple brightness rule would confuse the harbor with vessels.

**Extension:** `scripts/fetch_sar_window.py` extracts another small calibrated window locally. It uses range reads, never loads a full gigabyte-scale scene, and records provenance. New extractions change the evidence; retain the original file and its citation when comparing.